# AF Rotor Mechanisms Research Notebook

**Audience:** Cardiology researchers  
**Purpose:** Provide a reproducible Discovery workflow to generate, rank, and refine atrial fibrillation (AF) rotor mechanism research topics from an indexed literature shelf.

## 1. Workflow Outcomes

By the end of this notebook, the audience will see how to:
1. Query the indexed cardiology shelf with mechanism-specific prompts.
2. Convert retrieved evidence into testable hypotheses.
3. Generate five paper-ready topic candidates.
4. Score and prioritize topics by novelty, feasibility, and translational impact.
5. Define a concrete 12-month execution plan for top candidates.

## 1A. Practical Use Notes

Use this notebook as the local analysis layer after Discovery retrieval is complete:

1. Run the configuration cell and confirm the shelf name and indexed count.
2. Use the retrieval prompts in Discovery chat and paste conclusions into the evidence matrix.
3. Adjust topic candidates or rubric scores if the retrieved evidence changes.
4. Treat the ranked output as a decision aid, not as a substitute for expert review.
5. Use the top-2 summary as the handoff into protocol drafting or meeting discussion.

## 2. Architecture Of The Workflow

```mermaid
flowchart LR
    A[Knowledge Folder PDFs] --> B[Bookshelf Index]
    B --> C[Mechanism-Specific Retrieval]
    C --> D[Evidence Matrix]
    D --> E[Hypothesis Generation]
    E --> F[Topic Scoring]
    F --> G[Top 2 Study Designs]
```

## 3. Study Focus And Mechanistic Scope

We focus on plausible AF rotor-maintenance mechanisms:
- Fibrosis topology and conduction heterogeneity
- Fiber anisotropy gradients
- Calcium handling instability and alternans
- Autonomic gradients and refractoriness dispersion
- Inflammation-mediated coupling changes

Each mechanism should produce at least one **falsifiable hypothesis** and a measurable endpoint.

In [36]:
# 4. Workflow configuration
import pandas as pd
from datetime import datetime, timezone

SHELF_NAME = "cardiology-canon"
TOTAL_PDFS_IN_KNOWLEDGE = 508

# Optional: update this from the latest Discovery shelf status before each run
CURRENT_INDEX_COUNT = 545
RUN_METADATA = {
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "shelf_name": SHELF_NAME,
    "knowledge_pdf_count": TOTAL_PDFS_IN_KNOWLEDGE,
    "indexed_document_count": CURRENT_INDEX_COUNT,
    "index_coverage_ratio": round(
        CURRENT_INDEX_COUNT / max(TOTAL_PDFS_IN_KNOWLEDGE, 1), 3
    ),
}

print("Shelf:", SHELF_NAME)
print("PDFs in knowledge folder:", TOTAL_PDFS_IN_KNOWLEDGE)
print("Current indexed document count:", CURRENT_INDEX_COUNT)
print("Run timestamp (UTC):", RUN_METADATA["run_timestamp_utc"])
print("Index coverage ratio:", RUN_METADATA["index_coverage_ratio"])

Shelf: cardiology-canon
PDFs in knowledge folder: 508
Current indexed document count: 545
Run timestamp (UTC): 2026-07-22T04:00:23.155891+00:00
Index coverage ratio: 1.073


### How To Read This Output

- This printout confirms the active shelf, corpus size assumption, and current indexed count used in this run.
- The index coverage ratio is a quick sanity check that indexing is complete enough for retrieval.
- If counts look stale, update CURRENT_INDEX_COUNT before continuing so run metadata remains trustworthy.

In [37]:
# 5. Define the research brief that will drive Discovery retrieval
research_brief = {
    "project_title": "AF rotor mechanisms prioritization",
    "research_question": "Which AF rotor-maintenance mechanisms are best supported for a near-term translational study program?",
    "target_population": "Adults with persistent or high-burden atrial fibrillation",
    "decision_goal": "Select the top two mechanism-driven study concepts for protocol drafting.",
    "target_outputs": [
        "evidence matrix",
        "ranked topic list",
        "top-2 study blueprint",
        "submission-ready draft starter",
    ],
    "decision_criteria": [
        "mechanistic plausibility",
        "novelty",
        "12-month feasibility",
        "clinical impact",
        "data or mapping accessibility",
    ],
}

mechanisms_of_interest = [
    "Fibrosis topology",
    "Anisotropy gradients",
    "Calcium instability",
    "Autonomic heterogeneity",
    "Inflammation/coupling",
]

research_brief

{'project_title': 'AF rotor mechanisms prioritization',
 'research_question': 'Which AF rotor-maintenance mechanisms are best supported for a near-term translational study program?',
 'target_population': 'Adults with persistent or high-burden atrial fibrillation',
 'decision_goal': 'Select the top two mechanism-driven study concepts for protocol drafting.',
 'target_outputs': ['evidence matrix',
  'ranked topic list',
  'top-2 study blueprint',
  'submission-ready draft starter'],
 'decision_criteria': ['mechanistic plausibility',
  'novelty',
  '12-month feasibility',
  'clinical impact',
  'data or mapping accessibility']}

### How To Read This Output

- This dictionary is the study brief that anchors all downstream prompts and scoring decisions.
- Treat these fields as project controls: if the research question changes, update this object first and rerun from prompt generation onward.
- The mechanisms_of_interest list determines which rows are created in retrieval and evidence tracking tables.

## 6. Research Brief Context

This section defines the research question, decision criteria, and target outputs before any Discovery retrieval occurs.

The code cells below generate a Discovery-facing prompt packet and a retrieval capture scaffold so the workflow is driven from the notebook rather than from ad hoc chat prompts.

## 7. Retrieval Prompts To Use In Discovery Chat

Use these prompts directly in Discovery chat with your shelf. Keep prompts narrow to avoid generic AF anticoagulation retrieval.

### Prompt A: Evidence
From the indexed cardiology shelf, summarize evidence that AF rotors are stabilized by fibrosis topology, anisotropy gradients, calcium alternans, autonomic heterogeneity, and inflammation. For each mechanism return strongest support and strongest conflicting evidence.

### Prompt B: Gaps
For each rotor mechanism above, list unresolved and testable questions suitable for original studies, including one primary endpoint and expected direction of effect.

### Prompt C: Translation
Map each mechanism to one feasible human mapping study and one translational model study, with likely confounders and mitigation steps.

In [38]:
# 8. Initialize the retrieval capture scaffold that feeds the evidence matrix
retrieval_findings = {
    mechanism: {
        "supporting_evidence": [],
        "conflicting_evidence": [],
        "key_gap": "",
        "human_study_design": "",
        "translational_study_design": "",
        "confounders": [],
        "mitigations": [],
        "source_prompts_completed": [],
        "retrieval_timestamp_utc": "",
    }
    for mechanism in mechanisms_of_interest
}

retrieval_capture_sheet = pd.DataFrame([
    {
        "mechanism": mechanism,
        "prompt_name": prompt_name,
        "status": "pending",
        "source_title": "",
        "source_id": "",
        "chunk_id": "",
        "direct_quote": "",
        "confidence_0_to_1": None,
        "notes": "",
    }
    for mechanism in mechanisms_of_interest
    for prompt_name in ["Evidence", "Gap", "Translation"]
])

retrieval_capture_sheet

,mechanism,prompt_name,status,source_title,source_id,chunk_id,direct_quote,confidence_0_to_1,notes
0,Fibrosis topology,Evidence,pending,,,,,None,
1,Fibrosis topology,Gap,pending,,,,,None,
2,Fibrosis topology,Translation,pending,,,,,None,
3,Anisotropy gradients,Evidence,pending,,,,,None,
4,Anisotropy gradients,Gap,pending,,,,,None,
5,Anisotropy gradients,Translation,pending,,,,,None,
6,Calcium instability,Evidence,pending,,,,,None,
7,Calcium instability,Gap,pending,,,,,None,
8,Calcium instability,Translation,pending,,,,,None,
9,Autonomic heterogeneity,Evidence,pending,,,,,None,


### How To Read This Output

- Each row is one retrieval task: mechanism x prompt type (Evidence, Gap, Translation).
- Status starts as pending and should be updated as findings are captured.
- Source and quote fields are for provenance, enabling traceable claims in the evidence matrix and final package.

## Slow-Query Operating Mode (GraphRAG)

Use this section when retrieval runs are slow or spread across sessions.

Workflow:
1. Update retrieval statuses as queries complete (`pending`, `partial`, `complete`).
2. Run the progress tracker to see completion by mechanism.
3. Run the readiness check to know whether ranking is evidence-safe.
4. Save a resume checkpoint before ending a session so the next pass starts from a clear queue.

In [58]:
# Slow Mode A: Progress tracker by mechanism
prompt_order = ["Evidence", "Gap", "Translation"]

if "retrieval_capture_sheet" not in globals():
    retrieval_capture_sheet = pd.DataFrame([
        {
            "mechanism": mechanism,
            "prompt_name": prompt_name,
            "status": "pending",
            "source_title": "",
            "source_id": "",
            "chunk_id": "",
            "direct_quote": "",
            "confidence_0_to_1": None,
            "notes": "",
        }
        for mechanism in mechanisms_of_interest
        for prompt_name in prompt_order
    ])

status_norm = retrieval_capture_sheet.copy()
status_norm["status"] = (
    status_norm["status"]
    .fillna("pending")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"done": "complete", "completed": "complete", "in-progress": "partial"})
)

status_counts = (
    status_norm
    .groupby(["mechanism", "status"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["pending", "partial", "complete"], fill_value=0)
    .reset_index()
)
status_counts["total_prompts"] = status_counts[["pending", "partial", "complete"]].sum(axis=1)
status_counts["completion_pct"] = (
    100 * (status_counts["complete"] + 0.5 * status_counts["partial"]) / status_counts["total_prompts"].clip(lower=1)
).round(1)

overall_completion_pct = round(
    100 * (status_counts["complete"].sum() + 0.5 * status_counts["partial"].sum()) / status_counts["total_prompts"].sum(),
    1,
)

print(f"Overall retrieval progress: {overall_completion_pct}%")
status_counts.sort_values(["completion_pct", "mechanism"], ascending=[False, True])

Overall retrieval progress: 70.0%


status,mechanism,pending,partial,complete,total_prompts,completion_pct
3,Fibrosis topology,0,0,3,3,100.0
4,Inflammation/coupling,0,0,3,3,100.0
0,Anisotropy gradients,1,1,1,3,50.0
1,Autonomic heterogeneity,1,1,1,3,50.0
2,Calcium instability,1,1,1,3,50.0


In [40]:
# Slow Mode B: Safe-to-rank readiness check
if "evidence_matrix" not in globals():
    print("evidence_matrix not found. Run the evidence matrix cell first.")
    readiness = pd.DataFrame()
else:
    status_view = retrieval_capture_sheet.copy() if "retrieval_capture_sheet" in globals() else pd.DataFrame()
    if not status_view.empty:
        status_view["status"] = status_view["status"].fillna("pending").astype(str).str.lower()
        complete_by_mechanism = (
            status_view.assign(is_complete=status_view["status"].eq("complete"))
            .groupby("mechanism")["is_complete"]
            .sum()
            .to_dict()
        )
    else:
        complete_by_mechanism = {}

    readiness = evidence_matrix[[
        "Mechanism",
        "Supporting count",
        "Citation coverage",
        "Needs requery",
    ]].copy()
    readiness["complete_prompts"] = readiness["Mechanism"].map(complete_by_mechanism).fillna(0).astype(int)
    readiness["status_gate_pass"] = readiness["complete_prompts"] >= 3
    readiness["evidence_gate_pass"] = (
        (readiness["Supporting count"] >= 1)
        & (readiness["Citation coverage"] >= 0.5)
        & (~readiness["Needs requery"])
    )
    readiness["ready"] = readiness["status_gate_pass"] & readiness["evidence_gate_pass"]

    def _blockers(row):
        blockers = []
        if not row["status_gate_pass"]:
            blockers.append("complete all 3 prompts")
        if row["Supporting count"] < 1:
            blockers.append("add supporting evidence")
        if row["Citation coverage"] < 0.5:
            blockers.append("add citations/chunk refs")
        if row["Needs requery"]:
            blockers.append("resolve requery flag")
        return "; ".join(blockers)

    readiness["blockers"] = readiness.apply(_blockers, axis=1)
    ready_to_rank = bool(readiness["ready"].all())
    print("Safe-to-rank:", "YES" if ready_to_rank else "NO")
    readiness

Safe-to-rank: NO


In [41]:
# Slow Mode C: Resume checkpoint snapshot
from datetime import datetime, timezone

checkpoint_time = datetime.now(timezone.utc).isoformat()
status_view = retrieval_capture_sheet.copy() if "retrieval_capture_sheet" in globals() else pd.DataFrame()

if not status_view.empty:
    status_view["status"] = status_view["status"].fillna("pending").astype(str).str.lower()
    outstanding = status_view[status_view["status"].isin(["pending", "partial"])].copy()
else:
    outstanding = pd.DataFrame(columns=["mechanism", "prompt_name", "status", "notes"])

current_completion = globals().get("overall_completion_pct", None)
if current_completion is not None:
    current_completion = float(current_completion)

checkpoint = {
    "checkpoint_utc": checkpoint_time,
    "overall_retrieval_completion_pct": current_completion,
    "outstanding_prompt_count": int(len(outstanding)),
    "requery_count": int(len(globals().get("requery_queue", pd.DataFrame()))),
    "safe_to_rank": bool(globals().get("ready_to_rank", False)),
}

print("Checkpoint:", checkpoint)
outstanding[[col for col in ["mechanism", "prompt_name", "status", "notes"] if col in outstanding.columns]].head(20)

Checkpoint: {'checkpoint_utc': '2026-07-22T04:00:39.474836+00:00', 'overall_retrieval_completion_pct': 0.0, 'outstanding_prompt_count': 15, 'requery_count': 5, 'safe_to_rank': False}


,mechanism,prompt_name,status,notes
0,Fibrosis topology,Evidence,pending,
1,Fibrosis topology,Gap,pending,
2,Fibrosis topology,Translation,pending,
3,Anisotropy gradients,Evidence,pending,
4,Anisotropy gradients,Gap,pending,
5,Anisotropy gradients,Translation,pending,
6,Calcium instability,Evidence,pending,
7,Calcium instability,Gap,pending,
8,Calcium instability,Translation,pending,
9,Autonomic heterogeneity,Evidence,pending,


In [42]:
# Slow Mode D: Append checkpoint to local JSONL log
import json
from pathlib import Path

checkpoint_log_path = Path("retrieval-checkpoints.jsonl")

if "checkpoint" not in globals():
    raise RuntimeError("checkpoint not found. Run Slow Mode C first.")

with checkpoint_log_path.open("a", encoding="utf-8") as f:
    f.write(json.dumps(checkpoint, ensure_ascii=True) + "\n")

print(f"Checkpoint appended to: {checkpoint_log_path.resolve()}")
checkpoint

Checkpoint appended to: C:\source\cardiologycanon2026\evaluation\retrieval-checkpoints.jsonl


{'checkpoint_utc': '2026-07-22T04:00:39.474836+00:00',
 'overall_retrieval_completion_pct': 0.0,
 'outstanding_prompt_count': 15,
 'requery_count': 5,
 'safe_to_rank': False}

### Slow Mode D: Save Checkpoint To JSON

Run this cell to append the current checkpoint to a local JSONL log for progress tracking across sessions.

In [43]:
# 9. Render a copy-ready Discovery prompt packet
from IPython.display import Markdown, display

packet_prompt_library = globals().get("prompt_library", [
    {
        "prompt_name": "Evidence",
        "goal": "Gather strongest supporting and conflicting evidence for each mechanism.",
        "prompt_template": "From the indexed {shelf_name} shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: {mechanism}. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.",
    },
    {
        "prompt_name": "Gap",
        "goal": "Identify unresolved and testable questions.",
        "prompt_template": "Using the indexed {shelf_name} shelf, identify the most important unresolved and testable research gap for the mechanism: {mechanism}. Include one primary endpoint and expected direction of effect.",
    },
    {
        "prompt_name": "Translation",
        "goal": "Map mechanism to feasible study designs.",
        "prompt_template": "Using the indexed {shelf_name} shelf, map the mechanism: {mechanism} to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.",
    },
])

if "discovery_query_plan" not in globals():
    discovery_query_plan = pd.DataFrame([
        {
            "mechanism": mechanism,
            "prompt_name": prompt["prompt_name"],
            "goal": prompt["goal"],
            "prompt": prompt["prompt_template"].format(shelf_name=SHELF_NAME, mechanism=mechanism),
        }
        for mechanism in mechanisms_of_interest
        for prompt in packet_prompt_library
    ])

prompt_sections = [f"# Discovery Prompt Packet\n\nProject: {research_brief['project_title']}"]
for mechanism in mechanisms_of_interest:
    prompt_sections.append(f"## {mechanism}")
    mechanism_prompts = discovery_query_plan[discovery_query_plan["mechanism"] == mechanism]
    for _, row in mechanism_prompts.iterrows():
        prompt_sections.append(f"### {row['prompt_name']}")
        prompt_sections.append(row["prompt"])

discovery_prompt_packet = "\n\n".join(prompt_sections)
display(Markdown(discovery_prompt_packet))

# Discovery Prompt Packet

Project: AF rotor mechanisms prioritization

## Fibrosis topology

### Evidence

From the indexed cardiology-canon shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: Fibrosis topology. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.

### Gap

Using the indexed cardiology-canon shelf, identify the most important unresolved and testable research gap for the mechanism: Fibrosis topology. Include one primary endpoint and expected direction of effect.

### Translation

Using the indexed cardiology-canon shelf, map the mechanism: Fibrosis topology to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.

## Anisotropy gradients

### Evidence

From the indexed cardiology-canon shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: Anisotropy gradients. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.

### Gap

Using the indexed cardiology-canon shelf, identify the most important unresolved and testable research gap for the mechanism: Anisotropy gradients. Include one primary endpoint and expected direction of effect.

### Translation

Using the indexed cardiology-canon shelf, map the mechanism: Anisotropy gradients to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.

## Calcium instability

### Evidence

From the indexed cardiology-canon shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: Calcium instability. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.

### Gap

Using the indexed cardiology-canon shelf, identify the most important unresolved and testable research gap for the mechanism: Calcium instability. Include one primary endpoint and expected direction of effect.

### Translation

Using the indexed cardiology-canon shelf, map the mechanism: Calcium instability to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.

## Autonomic heterogeneity

### Evidence

From the indexed cardiology-canon shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: Autonomic heterogeneity. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.

### Gap

Using the indexed cardiology-canon shelf, identify the most important unresolved and testable research gap for the mechanism: Autonomic heterogeneity. Include one primary endpoint and expected direction of effect.

### Translation

Using the indexed cardiology-canon shelf, map the mechanism: Autonomic heterogeneity to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.

## Inflammation/coupling

### Evidence

From the indexed cardiology-canon shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: Inflammation/coupling. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.

### Gap

Using the indexed cardiology-canon shelf, identify the most important unresolved and testable research gap for the mechanism: Inflammation/coupling. Include one primary endpoint and expected direction of effect.

### Translation

Using the indexed cardiology-canon shelf, map the mechanism: Inflammation/coupling to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.

In [44]:
# 10. Generate the Discovery query plan from the research brief
prompt_library = [
    {
        "prompt_name": "Evidence",
        "goal": "Gather strongest supporting and conflicting evidence for each mechanism.",
        "prompt_template": "From the indexed {shelf_name} shelf, summarize strongest supporting evidence and strongest conflicting evidence for the mechanism: {mechanism}. Focus on rotor maintenance, mapping findings, substrate relationships, and translational relevance.",
    },
    {
        "prompt_name": "Gap",
        "goal": "Identify unresolved and testable questions.",
        "prompt_template": "Using the indexed {shelf_name} shelf, identify the most important unresolved and testable research gap for the mechanism: {mechanism}. Include one primary endpoint and expected direction of effect.",
    },
    {
        "prompt_name": "Translation",
        "goal": "Map mechanism to feasible study designs.",
        "prompt_template": "Using the indexed {shelf_name} shelf, map the mechanism: {mechanism} to one feasible human study design and one translational model design. Include likely confounders and mitigation steps.",
    },
]

discovery_query_plan = pd.DataFrame([
    {
        "mechanism": mechanism,
        "prompt_name": prompt["prompt_name"],
        "goal": prompt["goal"],
        "prompt": prompt["prompt_template"].format(shelf_name=SHELF_NAME, mechanism=mechanism),
    }
    for mechanism in mechanisms_of_interest
    for prompt in prompt_library
])

discovery_query_plan

,mechanism,prompt_name,goal,prompt
0,Fibrosis topology,Evidence,Gather strongest supporting and conflicting ev...,"From the indexed cardiology-canon shelf, summa..."
1,Fibrosis topology,Gap,Identify unresolved and testable questions.,"Using the indexed cardiology-canon shelf, iden..."
2,Fibrosis topology,Translation,Map mechanism to feasible study designs.,"Using the indexed cardiology-canon shelf, map ..."
3,Anisotropy gradients,Evidence,Gather strongest supporting and conflicting ev...,"From the indexed cardiology-canon shelf, summa..."
4,Anisotropy gradients,Gap,Identify unresolved and testable questions.,"Using the indexed cardiology-canon shelf, iden..."
5,Anisotropy gradients,Translation,Map mechanism to feasible study designs.,"Using the indexed cardiology-canon shelf, map ..."
6,Calcium instability,Evidence,Gather strongest supporting and conflicting ev...,"From the indexed cardiology-canon shelf, summa..."
7,Calcium instability,Gap,Identify unresolved and testable questions.,"Using the indexed cardiology-canon shelf, iden..."
8,Calcium instability,Translation,Map mechanism to feasible study designs.,"Using the indexed cardiology-canon shelf, map ..."
9,Autonomic heterogeneity,Evidence,Gather strongest supporting and conflicting ev...,"From the indexed cardiology-canon shelf, summa..."


In [45]:
# 11. Topic candidates generated from the mechanism framework
topics = pd.DataFrame([
    {
        "topic": "Fibrosis microarchitecture as rotor anchoring substrate in persistent AF",
        "hypothesis": "Patchy intermediate-density fibrosis stabilizes rotor cores more than diffuse fibrosis.",
        "primary_endpoint": "Rotor core dwell time by fibrosis class",
        "design": "LGE-MRI + high-density electroanatomic mapping"
    },
    {
        "topic": "Anisotropy gradients drive rotor drift toward ablation-resistant regions",
        "hypothesis": "Local anisotropy gradients predict rotor meander and recurrence better than conduction velocity alone.",
        "primary_endpoint": "Distance between predicted gradient zones and recurrent driver sites",
        "design": "Patient-specific substrate modeling + intraprocedural mapping"
    },
    {
        "topic": "Calcium alternans as a bridge from triggers to sustained rotor dynamics",
        "hypothesis": "Increasing calcium instability increases wavebreak-to-rotor conversion probability.",
        "primary_endpoint": "Wavebreak-to-rotor conversion rate",
        "design": "Human tissue optical mapping + ionic computational modeling"
    },
    {
        "topic": "Autonomic heterogeneity and dominant-frequency hierarchy at rotor sources",
        "hypothesis": "Autonomic gradients create local refractory dispersion that stabilizes high-frequency rotor sources.",
        "primary_endpoint": "Change in dominant frequency dispersion under autonomic modulation",
        "design": "AF mapping with controlled autonomic provocation"
    },
    {
        "topic": "Inflammation-linked connexin remodeling and transient rotor stabilization",
        "hypothesis": "Inflammatory surges alter coupling and transiently increase rotor persistence.",
        "primary_endpoint": "Association between biomarker surges and mapped rotor persistence",
        "design": "Prospective cohort with serial biomarkers + repeat mapping"
    }
])

topics

,topic,hypothesis,primary_endpoint,design
0,Fibrosis microarchitecture as rotor anchoring ...,Patchy intermediate-density fibrosis stabilize...,Rotor core dwell time by fibrosis class,LGE-MRI + high-density electroanatomic mapping
1,Anisotropy gradients drive rotor drift toward ...,Local anisotropy gradients predict rotor meand...,Distance between predicted gradient zones and ...,Patient-specific substrate modeling + intrapro...
2,Calcium alternans as a bridge from triggers to...,Increasing calcium instability increases waveb...,Wavebreak-to-rotor conversion rate,Human tissue optical mapping + ionic computati...
3,Autonomic heterogeneity and dominant-frequency...,Autonomic gradients create local refractory di...,Change in dominant frequency dispersion under ...,AF mapping with controlled autonomic provocation
4,Inflammation-linked connexin remodeling and tr...,Inflammatory surges alter coupling and transie...,Association between biomarker surges and mappe...,Prospective cohort with serial biomarkers + re...


### How To Read This Output

- This table is the candidate-topic backbone for prioritization.
- Each row should represent a testable mechanism with a hypothesis, one primary endpoint, and a feasible design concept.
- If retrieval evidence shifts, revise these rows before rescoring so ranking reflects current evidence.

## 12. Topic Scoring Rubric

Score each topic from 1 to 5 for:
1. **Mechanistic plausibility**
2. **Novelty**
3. **Feasibility in 12 months**
4. **Clinical impact**
5. **Data/mapping accessibility**

Weighted total:
$$
	ext{Score} = 0.25P + 0.20N + 0.20F + 0.25I + 0.10D
$$

In [46]:
# 13. Enter scores and rank topics
scores = pd.DataFrame([
    {"topic": topics.loc[0, "topic"], "P": 5, "N": 4, "F": 3, "I": 5, "D": 4},
    {"topic": topics.loc[1, "topic"], "P": 4, "N": 5, "F": 3, "I": 4, "D": 3},
    {"topic": topics.loc[2, "topic"], "P": 5, "N": 4, "F": 2, "I": 4, "D": 2},
    {"topic": topics.loc[3, "topic"], "P": 4, "N": 4, "F": 4, "I": 4, "D": 4},
    {"topic": topics.loc[4, "topic"], "P": 4, "N": 5, "F": 4, "I": 4, "D": 4},
])

weights = {"P": 0.25, "N": 0.20, "F": 0.20, "I": 0.25, "D": 0.10}

scores["weighted_score"] = sum(weights[column] * scores[column] for column in weights)
scores["weighted_score"] = scores["weighted_score"].round(2)

ranking = scores.sort_values(["weighted_score", "F", "I", "D"], ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.index.name = "rank"
ranking["gap_from_top"] = (ranking.loc[1, "weighted_score"] - ranking["weighted_score"]).round(2)

ranking

,topic,P,N,F,I,D,weighted_score,gap_from_top
rank,,,,,,,,
1,Fibrosis microarchitecture as rotor anchoring ...,5,4,3,5,4,4.30,0.00
2,Inflammation-linked connexin remodeling and tr...,4,5,4,4,4,4.20,0.10
3,Autonomic heterogeneity and dominant-frequency...,4,4,4,4,4,4.00,0.30
4,Anisotropy gradients drive rotor drift toward ...,4,5,3,4,3,3.90,0.40
5,Calcium alternans as a bridge from triggers to...,5,4,2,4,2,3.65,0.65


### How To Read This Output

- weighted_score is the baseline rubric score before evidence-quality adjustments.
- gap_from_top shows how far each candidate is from the current leader.
- Use this as a draft prioritization view until evidence-gated ranking is computed later.

In [47]:
# 14. Summarize the top two candidates for discussion or protocol handoff
top_2_summary = (
    ranking.head(2)
    .merge(topics, on="topic", how="left")
    [["topic", "weighted_score", "gap_from_top", "hypothesis", "primary_endpoint", "design"]]
    .rename(columns={
        "weighted_score": "score",
        "gap_from_top": "score_gap",
        "primary_endpoint": "primary endpoint"
    })
)

top_2_summary

,topic,score,score_gap,hypothesis,primary endpoint,design
0,Fibrosis microarchitecture as rotor anchoring ...,4.3,0.0,Patchy intermediate-density fibrosis stabilize...,Rotor core dwell time by fibrosis class,LGE-MRI + high-density electroanatomic mapping
1,Inflammation-linked connexin remodeling and tr...,4.2,0.1,Inflammatory surges alter coupling and transie...,Association between biomarker surges and mappe...,Prospective cohort with serial biomarkers + re...


In [48]:
# 15. Visual score dashboard
def make_bar(value, max_value, width=16, filled='█', empty='░'):
    filled_width = int(round((value / max_value) * width)) if max_value else 0
    return filled * filled_width + empty * (width - filled_width)

score_dashboard = ranking[["topic", "P", "N", "F", "I", "D", "weighted_score", "gap_from_top"]].copy()

for column in ["P", "N", "F", "I", "D"]:
    score_dashboard[f"{column}_visual"] = score_dashboard[column].apply(
        lambda value: f"{make_bar(value, 5)} {value}/5"
    )

score_dashboard["score_visual"] = score_dashboard["weighted_score"].apply(
    lambda value: f"{make_bar(value, 5)} {value:.2f}"
)
score_dashboard["gap_visual"] = score_dashboard["gap_from_top"].apply(
    lambda value: f"{make_bar(value, max(ranking['gap_from_top'].max(), 0.01))} {value:.2f}"
)

score_dashboard[[
    "topic",
    "score_visual",
    "gap_visual",
    "P_visual",
    "N_visual",
    "F_visual",
    "I_visual",
    "D_visual",
]]

,topic,score_visual,gap_visual,P_visual,N_visual,F_visual,I_visual,D_visual
rank,,,,,,,,
1,Fibrosis microarchitecture as rotor anchoring ...,██████████████░░ 4.30,░░░░░░░░░░░░░░░░ 0.00,████████████████ 5/5,█████████████░░░ 4/5,██████████░░░░░░ 3/5,████████████████ 5/5,█████████████░░░ 4/5
2,Inflammation-linked connexin remodeling and tr...,█████████████░░░ 4.20,██░░░░░░░░░░░░░░ 0.10,█████████████░░░ 4/5,████████████████ 5/5,█████████████░░░ 4/5,█████████████░░░ 4/5,█████████████░░░ 4/5
3,Autonomic heterogeneity and dominant-frequency...,█████████████░░░ 4.00,███████░░░░░░░░░ 0.30,█████████████░░░ 4/5,█████████████░░░ 4/5,█████████████░░░ 4/5,█████████████░░░ 4/5,█████████████░░░ 4/5
4,Anisotropy gradients drive rotor drift toward ...,████████████░░░░ 3.90,██████████░░░░░░ 0.40,█████████████░░░ 4/5,████████████████ 5/5,██████████░░░░░░ 3/5,█████████████░░░ 4/5,██████████░░░░░░ 3/5
5,Calcium alternans as a bridge from triggers to...,████████████░░░░ 3.65,████████████████ 0.65,████████████████ 5/5,█████████████░░░ 4/5,██████░░░░░░░░░░ 2/5,█████████████░░░ 4/5,██████░░░░░░░░░░ 2/5


## 16. Evidence Matrix

This section now derives the matrix structure from earlier notebook outputs, especially the topic table created above.

What is auto-populated from previous cells:

- mechanism label
- matched topic candidate
- working hypothesis
- proposed endpoint
- a generated default gap statement

What is **not** invented by the notebook:

- supporting evidence text
- conflicting evidence text

Those evidence fields remain an optional `retrieval_findings` input, because they must come from actual Discovery retrieval results rather than from hard-coded placeholder claims.

In [57]:
# 15b. Load REAL GraphRAG retrieval results into the pipeline (agent -> JSON -> notebook bridge)
#
# The Jupyter kernel cannot call the Discovery GraphRAG shelf directly: `cardiology-canon`
# is exposed only as a stdio MCP tool available to the Copilot agent, not over an HTTP
# endpoint the kernel can reach. So genuine queries are executed by the agent against the
# shelf (bookshelf.search / bookshelf.ask) and the results are persisted to
# `retrieval-findings.json`. This cell loads that real output and drives the evidence matrix.
#
# IMPORTANT (honesty): the results below are exactly what the corpus returned. The
# `cardiology-canon` shelf is clinical-guideline / outcome-trial material, so only
# "Fibrosis topology" and "Inflammation/coupling" could be substantiated (at a guideline /
# association level). The three mechanistic hypotheses returned no direct evidence and are
# recorded as unsupported, which is what keeps the evidence gate correctly blocked.
import json
from pathlib import Path
from datetime import datetime, timezone

_findings_path = Path.cwd() / "retrieval-findings.json"
if not _findings_path.exists():
    _findings_path = Path("retrieval-findings.json").resolve()

print(f"Loading real GraphRAG retrieval results from: {_findings_path}")
_raw_findings = json.loads(_findings_path.read_text(encoding="utf-8"))
_retrieval_meta = _raw_findings.pop("_meta", {})

# Merge the loaded results into the existing retrieval_findings scaffold (preserve schema keys).
for _mech in mechanisms_of_interest:
    _loaded = _raw_findings.get(_mech, {})
    _target = retrieval_findings.setdefault(_mech, {})
    for _key, _value in _loaded.items():
        _target[_key] = _value
    _target.setdefault("supporting_evidence", [])
    _target.setdefault("conflicting_evidence", [])
    _target.setdefault("confounders", [])

# Update the capture sheet status honestly from what retrieval actually returned.
def _status_for(prompt_name, entry):
    has_support = bool(entry.get("supporting_evidence"))
    has_any = has_support or bool(entry.get("conflicting_evidence"))
    has_gap = bool(entry.get("key_gap"))
    if prompt_name == "Evidence":
        return "complete" if has_support else ("partial" if has_gap or has_any else "pending")
    if prompt_name == "Gap":
        return "complete" if has_gap else "pending"
    if prompt_name == "Translation":
        return "complete" if has_support else "pending"
    return "pending"

for _idx, _row in retrieval_capture_sheet.iterrows():
    _mech = _row["mechanism"]
    _entry = retrieval_findings.get(_mech, {})
    retrieval_capture_sheet.at[_idx, "status"] = _status_for(_row["prompt_name"], _entry)
    if _row["prompt_name"] == "Evidence":
        _support = _entry.get("supporting_evidence") or []
        if _support:
            _first = _support[0]
            retrieval_capture_sheet.at[_idx, "source_title"] = _first.get("source_title", "")
            retrieval_capture_sheet.at[_idx, "source_id"] = _first.get("source_id", "")
            retrieval_capture_sheet.at[_idx, "chunk_id"] = _first.get("chunk_id", "")
            retrieval_capture_sheet.at[_idx, "direct_quote"] = _first.get("direct_quote", "")
            retrieval_capture_sheet.at[_idx, "confidence_0_to_1"] = _first.get("confidence")
            retrieval_capture_sheet.at[_idx, "notes"] = f"{len(_support)} supporting item(s) from shelf"
        else:
            retrieval_capture_sheet.at[_idx, "notes"] = _entry.get("key_gap", "No direct evidence in corpus")

_supported = [m for m in mechanisms_of_interest if retrieval_findings.get(m, {}).get("supporting_evidence")]
_unsupported = [m for m in mechanisms_of_interest if not retrieval_findings.get(m, {}).get("supporting_evidence")]

print(f"\nShelf: {_retrieval_meta.get('shelf_name', SHELF_NAME)} "
      f"(provider={_retrieval_meta.get('provider_id', 'graphrag-zero')})")
print(f"Retrieval method: {_retrieval_meta.get('retrieval_method', 'bookshelf.search + bookshelf.ask')}")
print(f"\nMechanisms WITH corpus support ({len(_supported)}): {_supported}")
print(f"Mechanisms with NO direct support ({len(_unsupported)}): {_unsupported}")
print("\nCapture-sheet status now reflects real retrieval output:")
retrieval_capture_sheet[["mechanism", "prompt_name", "status", "source_title", "confidence_0_to_1"]]


Loading real GraphRAG retrieval results from: c:\source\cardiologycanon2026\evaluation\retrieval-findings.json

Shelf: cardiology-canon (provider=graphrag-zero)
Retrieval method: bookshelf.search + bookshelf.ask

Mechanisms WITH corpus support (2): ['Fibrosis topology', 'Inflammation/coupling']
Mechanisms with NO direct support (3): ['Anisotropy gradients', 'Calcium instability', 'Autonomic heterogeneity']

Capture-sheet status now reflects real retrieval output:


,mechanism,prompt_name,status,source_title,confidence_0_to_1
0,Fibrosis topology,Evidence,complete,2020 ESC guideline for AF management and diagn...,0.55
1,Fibrosis topology,Gap,complete,,None
2,Fibrosis topology,Translation,complete,,None
3,Anisotropy gradients,Evidence,partial,,None
4,Anisotropy gradients,Gap,complete,,None
5,Anisotropy gradients,Translation,pending,,None
6,Calcium instability,Evidence,partial,,None
7,Calcium instability,Gap,complete,,None
8,Calcium instability,Translation,pending,,None
9,Autonomic heterogeneity,Evidence,partial,,None


In [59]:
# 17. Build the evidence matrix with provenance and quality controls
mechanism_config = {
    "Fibrosis topology": {"keywords": ["fibrosis"]},
    "Anisotropy gradients": {"keywords": ["anisotropy"]},
    "Calcium instability": {"keywords": ["calcium", "alternans"]},
    "Autonomic heterogeneity": {"keywords": ["autonomic", "dominant-frequency"]},
    "Inflammation/coupling": {"keywords": ["inflammation", "connexin"]},
}

# Optional external input: populate this from real Discovery retrieval output.
# Supported format for each evidence item:
# {"claim": "...", "source_title": "...", "source_id": "...", "chunk_id": "...",
#  "direct_quote": "...", "confidence": 0.0-1.0}
retrieval_findings = globals().get("retrieval_findings", {})

def _match_topic_row(keywords):
    pattern = "|".join(keywords)
    matches = topics[topics["topic"].str.contains(pattern, case=False, regex=True)]
    if not matches.empty:
        return matches.iloc[0]
    return None

def _default_gap(topic_row):
    if topic_row is None:
        return "Need gap statement from retrieved evidence."
    hypothesis = str(topic_row["hypothesis"]).rstrip(".")
    return f"Need direct evidence to test whether {hypothesis.lower()}."

def _normalize_evidence_items(mechanism, evidence_items, evidence_type):
    normalized = []
    for item in evidence_items or []:
        if isinstance(item, dict):
            normalized.append({
                "Mechanism": mechanism,
                "Evidence type": evidence_type,
                "Claim": item.get("claim", ""),
                "Source title": item.get("source_title", ""),
                "Source ID": item.get("source_id", ""),
                "Chunk ID": item.get("chunk_id", ""),
                "Direct quote": item.get("direct_quote", ""),
                "Confidence": item.get("confidence", None),
            })
        else:
            normalized.append({
                "Mechanism": mechanism,
                "Evidence type": evidence_type,
                "Claim": str(item),
                "Source title": "",
                "Source ID": "",
                "Chunk ID": "",
                "Direct quote": "",
                "Confidence": None,
            })
    return normalized

evidence_records = []
evidence_rows = []

for mechanism, config in mechanism_config.items():
    topic_row = _match_topic_row(config["keywords"])
    findings = retrieval_findings.get(mechanism, {})
    supporting_items = findings.get("supporting_evidence", [])
    conflicting_items = findings.get("conflicting_evidence", [])

    normalized_supporting = _normalize_evidence_items(
        mechanism, supporting_items, "supporting"
    )
    normalized_conflicting = _normalize_evidence_items(
        mechanism, conflicting_items, "conflicting"
    )
    mechanism_records = normalized_supporting + normalized_conflicting
    evidence_records.extend(mechanism_records)

    support_count = len(normalized_supporting)
    conflict_count = len(normalized_conflicting)
    cited_count = sum(
        1
        for record in mechanism_records
        if record["Source title"] or record["Source ID"] or record["Chunk ID"]
    )
    total_count = len(mechanism_records)
    citation_coverage = round(cited_count / total_count, 2) if total_count else 0.0

    confidence_values = [
        record["Confidence"]
        for record in mechanism_records
        if isinstance(record["Confidence"], (float, int))
    ]
    avg_confidence = round(sum(confidence_values) / len(confidence_values), 2) if confidence_values else None

    evidence_strength = min(
        5.0,
        round(
            1.5 + (0.8 * support_count) + (1.5 * citation_coverage),
            2,
        ),
    ) if support_count else 1.0
    conflict_penalty = min(2.0, round(conflict_count * 0.5, 2))

    needs_requery = (
        support_count < 1
        or citation_coverage < 0.5
        or (conflict_count > support_count and support_count > 0)
    )

    evidence_rows.append({
        "Mechanism": mechanism,
        "Candidate topic": topic_row["topic"] if topic_row is not None else "No matched topic candidate",
        "Working hypothesis": topic_row["hypothesis"] if topic_row is not None else "Add hypothesis",
        "Supporting evidence": "\n".join(
            [f"- {record['Claim']}" for record in normalized_supporting]
        ),
        "Conflicting evidence": "\n".join(
            [f"- {record['Claim']}" for record in normalized_conflicting]
        ),
        "Supporting count": support_count,
        "Conflicting count": conflict_count,
        "Citation coverage": citation_coverage,
        "Average confidence": avg_confidence,
        "Evidence strength (1-5)": evidence_strength,
        "Conflict penalty (0-2)": conflict_penalty,
        "Needs requery": needs_requery,
        "Requery prompt": findings.get("requery_prompt") or (
            "Re-query this mechanism with a citation-required prompt and include at least one direct quote."
            if needs_requery else ""
        ),
        "Key gap": findings.get("key_gap") or _default_gap(topic_row),
        "Proposed endpoint": topic_row["primary_endpoint"] if topic_row is not None else "Define endpoint from retrieved evidence",
        "Retrieval timestamp UTC": findings.get("retrieval_timestamp_utc", ""),
    })

evidence_matrix = pd.DataFrame(evidence_rows)
evidence_record_table = pd.DataFrame(evidence_records)
requery_queue = evidence_matrix[evidence_matrix["Needs requery"]].copy()

evidence_matrix

,Mechanism,Candidate topic,Working hypothesis,Supporting evidence,Conflicting evidence,Supporting count,Conflicting count,Citation coverage,Average confidence,Evidence strength (1-5),Conflict penalty (0-2),Needs requery,Requery prompt,Key gap,Proposed endpoint,Retrieval timestamp UTC
0,Fibrosis topology,Fibrosis microarchitecture as rotor anchoring ...,Patchy intermediate-density fibrosis stabilize...,- Enlarged atrial size and/or atrial fibrosis ...,- Randomized ablation trials show only modest ...,2,1,1.0,0.52,4.6,0.5,False,,Corpus provides guideline-level associations o...,Rotor core dwell time by fibrosis class,2025-01-01T00:00:00Z
1,Anisotropy gradients,Anisotropy gradients drive rotor drift toward ...,Local anisotropy gradients predict rotor meand...,,,0,0,0.0,NaN,1.0,0.0,True,Ingest electrophysiology literature on anisotr...,No direct evidence in cardiology-canon. Target...,Distance between predicted gradient zones and ...,2025-01-01T00:00:00Z
2,Calcium instability,Calcium alternans as a bridge from triggers to...,Increasing calcium instability increases waveb...,,,0,0,0.0,NaN,1.0,0.0,True,Ingest cellular-electrophysiology literature o...,No direct evidence in cardiology-canon. Search...,Wavebreak-to-rotor conversion rate,2025-01-01T00:00:00Z
3,Autonomic heterogeneity,Autonomic heterogeneity and dominant-frequency...,Autonomic gradients create local refractory di...,,,0,0,0.0,NaN,1.0,0.0,True,Ingest autonomic-nervous-system and ganglionat...,No direct evidence in cardiology-canon. Search...,Change in dominant frequency dispersion under ...,2025-01-01T00:00:00Z
4,Inflammation/coupling,Inflammation-linked connexin remodeling and tr...,Inflammatory surges alter coupling and transie...,- High-sensitivity C-reactive protein and stat...,- Anti-inflammatory/statin effects on AF are i...,2,1,1.0,0.50,4.6,0.5,False,,Corpus supports an epidemiologic inflammation-...,Association between biomarker surges and mappe...,2025-01-01T00:00:00Z


### How To Read This Output

- This matrix consolidates support, conflict, and traceability quality per mechanism.
- Evidence strength and conflict penalty are derived indicators used in the next ranking step.
- Needs requery equals True when evidence is insufficient or under-cited, signaling more Discovery retrieval is required.

## 18. Evidence-Gated Ranking And Requery Gate

This step updates ranking using evidence quality from the matrix above.

Gate rules before final prioritization:
1. At least one supporting item per mechanism
2. Citation coverage >= 0.5 for each mechanism
3. No mechanism where conflicts dominate supporting evidence

If any mechanism fails the gate, the notebook surfaces a requery queue and keeps output in a draft state.

In [60]:
# 19. Recompute ranking with evidence quality and enforce requery gate
evidence_join = evidence_matrix[[
    "Candidate topic",
    "Evidence strength (1-5)",
    "Conflict penalty (0-2)",
    "Needs requery",
]].copy()

scored_with_evidence = scores.merge(
    evidence_join,
    left_on="topic",
    right_on="Candidate topic",
    how="left",
).drop(columns=["Candidate topic"])

scored_with_evidence["Evidence strength (1-5)"] = scored_with_evidence[
    "Evidence strength (1-5)"
].fillna(1.0)
scored_with_evidence["Conflict penalty (0-2)"] = scored_with_evidence[
    "Conflict penalty (0-2)"
].fillna(0.0)
scored_with_evidence["Needs requery"] = scored_with_evidence["Needs requery"].fillna(True)

scored_with_evidence["base_weighted_score"] = scored_with_evidence["weighted_score"]
scored_with_evidence["evidence_bonus"] = (
    0.15 * scored_with_evidence["Evidence strength (1-5)"]
).round(2)
scored_with_evidence["conflict_penalty_adjustment"] = (
    0.2 * scored_with_evidence["Conflict penalty (0-2)"]
).round(2)

scored_with_evidence["weighted_score"] = (
    scored_with_evidence["base_weighted_score"]
    + scored_with_evidence["evidence_bonus"]
    - scored_with_evidence["conflict_penalty_adjustment"]
).round(2)

ranking = scored_with_evidence.sort_values(
    ["weighted_score", "F", "I", "D"], ascending=False
).reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.index.name = "rank"
ranking["gap_from_top"] = (
    ranking.iloc[0]["weighted_score"] - ranking["weighted_score"]
).round(2)

finalization_blocked = not requery_queue.empty
ranking["rank_status"] = "draft" if finalization_blocked else "final"

top_2_summary = (
    ranking.head(2)
    .merge(topics, on="topic", how="left")
    [[
        "topic",
        "weighted_score",
        "gap_from_top",
        "rank_status",
        "hypothesis",
        "primary_endpoint",
        "design",
    ]]
    .rename(columns={
        "weighted_score": "score",
        "gap_from_top": "score_gap",
        "primary_endpoint": "primary endpoint",
    })
)

if finalization_blocked:
    print("Requery required before final prioritization. Review requery_queue.")
    display(requery_queue[["Mechanism", "Requery prompt", "Citation coverage", "Supporting count", "Conflicting count"]])
else:
    print("All gate checks passed. Ranking is final.")

top_2_summary

Requery required before final prioritization. Review requery_queue.


,Mechanism,Requery prompt,Citation coverage,Supporting count,Conflicting count
1,Anisotropy gradients,Ingest electrophysiology literature on anisotr...,0.0,0,0
2,Calcium instability,Ingest cellular-electrophysiology literature o...,0.0,0,0
3,Autonomic heterogeneity,Ingest autonomic-nervous-system and ganglionat...,0.0,0,0


,topic,score,score_gap,rank_status,hypothesis,primary endpoint,design
0,Fibrosis microarchitecture as rotor anchoring ...,4.89,0.0,draft,Patchy intermediate-density fibrosis stabilize...,Rotor core dwell time by fibrosis class,LGE-MRI + high-density electroanatomic mapping
1,Inflammation-linked connexin remodeling and tr...,4.79,0.1,draft,Inflammatory surges alter coupling and transie...,Association between biomarker surges and mappe...,Prospective cohort with serial biomarkers + re...


### How To Read This Output

- rank_status equals draft when any mechanism still requires requery; it becomes final only after gate criteria pass.
- The displayed requery queue is your action list for targeted follow-up prompts.
- Re-run this cell after updating retrieval_findings to promote the ranking from draft to final.

## 20. Top-2 Proposal Blueprint

Use the `top_2_summary` output from the previous section as the handoff object for study design discussion.

For the two highest-ranked topics, define:
1. Population and eligibility
2. Mapping method and timing
3. Substrate or biomarker measurements
4. Primary and secondary endpoints
5. Prespecified confounders
6. Analysis strategy
7. Milestones at months 3, 6, 9, and 12

If two topics are close on score, use feasibility, data accessibility, and translational impact as explicit tie-breakers in the meeting discussion.

In [51]:
# 21. Twelve-month milestone planner for the top 2 topics
milestone_template = pd.DataFrame([
    {"month": 1, "milestone": "Protocol draft + ethics pre-submission"},
    {"month": 3, "milestone": "Final protocol + site readiness"},
    {"month": 6, "milestone": "First cohort recruited + pilot mapping quality check"},
    {"month": 9, "milestone": "Interim analysis of mechanistic endpoints"},
    {"month": 12, "milestone": "Abstract/manuscript draft for top candidate"}
])

top_2_topics = top_2_summary[["topic"]].copy()
top_2_topics["_merge_key"] = 1
milestone_template["_merge_key"] = 1

plan = (
    top_2_topics.merge(milestone_template, on="_merge_key")
    .drop(columns="_merge_key")
    .sort_values(["topic", "month"])
    .reset_index(drop=True)
)

plan

,topic,month,milestone
0,Fibrosis microarchitecture as rotor anchoring ...,1,Protocol draft + ethics pre-submission
1,Fibrosis microarchitecture as rotor anchoring ...,3,Final protocol + site readiness
2,Fibrosis microarchitecture as rotor anchoring ...,6,First cohort recruited + pilot mapping quality...
3,Fibrosis microarchitecture as rotor anchoring ...,9,Interim analysis of mechanistic endpoints
4,Fibrosis microarchitecture as rotor anchoring ...,12,Abstract/manuscript draft for top candidate
5,Inflammation-linked connexin remodeling and tr...,1,Protocol draft + ethics pre-submission
6,Inflammation-linked connexin remodeling and tr...,3,Final protocol + site readiness
7,Inflammation-linked connexin remodeling and tr...,6,First cohort recruited + pilot mapping quality...
8,Inflammation-linked connexin remodeling and tr...,9,Interim analysis of mechanistic endpoints
9,Inflammation-linked connexin remodeling and tr...,12,Abstract/manuscript draft for top candidate


## 22. Submission Package Starter

This section converts the top-ranked topic into a submission-oriented draft starter.

It is not a final grant or manuscript by itself. It is the packaging step that prepares:

- working title
- rationale and hypothesis
- specific aims
- study design summary
- endpoints and confounders
- milestone snapshot
- immediate next inputs needed for a real submission

In [61]:
# 23. Build a submission-ready draft starter for the top-ranked topic
from IPython.display import Markdown, display

top_candidate = top_2_summary.iloc[0].to_dict()
top_candidate_plan = plan[plan["topic"] == top_candidate["topic"]].copy()

matching_evidence = evidence_matrix[
    evidence_matrix["Candidate topic"] == top_candidate["topic"]
].copy()

if not matching_evidence.empty:
    evidence_row = matching_evidence.iloc[0].to_dict()
else:
    evidence_row = {
        "Mechanism": "",
        "Supporting evidence": "",
        "Conflicting evidence": "",
        "Key gap": "Need evidence synthesis from Discovery retrieval.",
    }

default_confounders = [
    "AF phenotype and chronicity",
    "prior ablation or antiarrhythmic exposure",
    "mapping density and signal quality",
    "imaging or biomarker acquisition timing",
]

retrieved_confounders = retrieval_findings.get(
    evidence_row.get("Mechanism", ""), {}
 ).get("confounders", [])

prespecified_confounders = retrieved_confounders or default_confounders

specific_aims = [
    "Aim 1: Validate the core mechanistic association in the target study population.",
    "Aim 2: Quantify the primary endpoint and test whether the proposed substrate signal predicts rotor behavior.",
    "Aim 3: Evaluate key confounders and define the translational path for a larger confirmatory study.",
]

submission_package = {
    "working_title": f"Prospective evaluation of {top_candidate['topic'].lower()}",
    "rationale": "Discovery retrieval and downstream ranking identified this topic as the highest-priority candidate based on mechanistic plausibility, feasibility, and translational impact.",
    "evidence_for": evidence_row.get("Supporting evidence", ""),
    "evidence_against": evidence_row.get("Conflicting evidence", ""),
    "central_hypothesis": top_candidate["hypothesis"],
    "specific_aims": specific_aims,
    "proposed_design": top_candidate["design"],
    "primary_endpoint": top_candidate["primary endpoint"],
    "key_gap": evidence_row.get("Key gap", ""),
    "suggested_secondary_endpoints": [
        "signal reproducibility across mapping sessions or analytic replicates",
        "association with short-term recurrence or mechanistic persistence",
        "incremental predictive value beyond standard substrate metrics",
    ],
    "prespecified_confounders": prespecified_confounders,
    "milestones": top_candidate_plan[["month", "milestone"]].to_dict("records"),
    "immediate_next_inputs": [
        "final inclusion and exclusion criteria",
        "sample size estimate",
        "site capability and workflow constraints",
        "statistical analysis plan",
        "ethics and governance requirements",
    ],
}

milestone_lines = "\n".join(
    f"- Month {row['month']}: {row['milestone']}"
    for row in submission_package["milestones"]
)

evidence_for_text = submission_package["evidence_for"] or "Add supporting evidence from Discovery retrieval."
evidence_against_text = submission_package["evidence_against"] or "Add conflicting evidence from Discovery retrieval."
confounder_lines = "\n".join(
    f"- {item}" for item in submission_package["prespecified_confounders"]
)

draft_markdown = f"""# Submission-Ready Draft Starter

## Working Title
{submission_package['working_title']}

## Rationale
{submission_package['rationale']}

## Evidence Supporting The Topic
{evidence_for_text}

## Evidence Arguing Against Or Limiting The Topic
{evidence_against_text}

## Central Hypothesis
{submission_package['central_hypothesis']}

## Specific Aims
1. {submission_package['specific_aims'][0]}
2. {submission_package['specific_aims'][1]}
3. {submission_package['specific_aims'][2]}

## Proposed Design
{submission_package['proposed_design']}

## Primary Endpoint
{submission_package['primary_endpoint']}

## Key Gap To Close
{submission_package['key_gap']}

## Suggested Secondary Endpoints
- {submission_package['suggested_secondary_endpoints'][0]}
- {submission_package['suggested_secondary_endpoints'][1]}
- {submission_package['suggested_secondary_endpoints'][2]}

## Prespecified Confounders
{confounder_lines}

## 12-Month Milestones
{milestone_lines}

## Immediate Next Inputs Needed For Submission
- {submission_package['immediate_next_inputs'][0]}
- {submission_package['immediate_next_inputs'][1]}
- {submission_package['immediate_next_inputs'][2]}
- {submission_package['immediate_next_inputs'][3]}
- {submission_package['immediate_next_inputs'][4]}
"""

display(Markdown(draft_markdown))

pd.DataFrame(
    {
        "field": list(submission_package.keys()),
        "value_preview": [
            submission_package["working_title"],
            submission_package["rationale"],
            submission_package["evidence_for"],
            submission_package["evidence_against"],
            submission_package["central_hypothesis"],
            "; ".join(submission_package["specific_aims"]),
            submission_package["proposed_design"],
            submission_package["primary_endpoint"],
            submission_package["key_gap"],
            "; ".join(submission_package["suggested_secondary_endpoints"]),
            "; ".join(submission_package["prespecified_confounders"]),
            f"{len(submission_package['milestones'])} milestones",
            "; ".join(submission_package["immediate_next_inputs"]),
        ],
    }
)

# Submission-Ready Draft Starter

## Working Title
Prospective evaluation of fibrosis microarchitecture as rotor anchoring substrate in persistent af

## Rationale
Discovery retrieval and downstream ranking identified this topic as the highest-priority candidate based on mechanistic plausibility, feasibility, and translational impact.

## Evidence Supporting The Topic
- Enlarged atrial size and/or atrial fibrosis mark patients at higher risk of AF recurrence and are relevant to ablation candidacy and substrate-directed, personalized rhythm control.
- Non-invasive substrate characterization (echocardiography/MRI/CT) alongside clinical factors is proposed to individualize rhythm-control strategy selection.

## Evidence Arguing Against Or Limiting The Topic
- Randomized ablation trials show only modest long-term reductions in AF burden, indicating that substrate/fibrosis targeting alone does not guarantee durable rhythm control.

## Central Hypothesis
Patchy intermediate-density fibrosis stabilizes rotor cores more than diffuse fibrosis.

## Specific Aims
1. Aim 1: Validate the core mechanistic association in the target study population.
2. Aim 2: Quantify the primary endpoint and test whether the proposed substrate signal predicts rotor behavior.
3. Aim 3: Evaluate key confounders and define the translational path for a larger confirmatory study.

## Proposed Design
LGE-MRI + high-density electroanatomic mapping

## Primary Endpoint
Rotor core dwell time by fibrosis class

## Key Gap To Close
Corpus provides guideline-level associations only; no mechanistic (LGE-MRI wavefront / conduction) evidence linking fibrosis topology to rotor anchoring.

## Suggested Secondary Endpoints
- signal reproducibility across mapping sessions or analytic replicates
- association with short-term recurrence or mechanistic persistence
- incremental predictive value beyond standard substrate metrics

## Prespecified Confounders
- AF phenotype and chronicity (paroxysmal vs persistent)
- Left atrial size / remodeling
- Imaging modality and thresholding used to quantify fibrosis

## 12-Month Milestones
- Month 1: Protocol draft + ethics pre-submission
- Month 3: Final protocol + site readiness
- Month 6: First cohort recruited + pilot mapping quality check
- Month 9: Interim analysis of mechanistic endpoints
- Month 12: Abstract/manuscript draft for top candidate

## Immediate Next Inputs Needed For Submission
- final inclusion and exclusion criteria
- sample size estimate
- site capability and workflow constraints
- statistical analysis plan
- ethics and governance requirements


,field,value_preview
0,working_title,Prospective evaluation of fibrosis microarchit...
1,rationale,Discovery retrieval and downstream ranking ide...
2,evidence_for,- Enlarged atrial size and/or atrial fibrosis ...
3,evidence_against,- Randomized ablation trials show only modest ...
4,central_hypothesis,Patchy intermediate-density fibrosis stabilize...
5,specific_aims,Aim 1: Validate the core mechanistic associati...
6,proposed_design,LGE-MRI + high-density electroanatomic mapping
7,primary_endpoint,Rotor core dwell time by fibrosis class
8,key_gap,Corpus provides guideline-level associations o...
9,suggested_secondary_endpoints,signal reproducibility across mapping sessions...


In [62]:
# Export final draft markdown for review (guarded by readiness thresholds)
from pathlib import Path

if "draft_markdown" not in globals():
    raise RuntimeError("draft_markdown not found. Run the submission package cell first.")

# Export guard: block accidental "final" exports when retrieval is incomplete.
# Set this to True only when you intentionally want to share a draft-state document.
FORCE_EXPORT_DRAFT = False
MIN_RETRIEVAL_PROGRESS_PCT = 80.0

current_progress_pct = float(globals().get("overall_completion_pct", 0.0) or 0.0)
current_ready_to_rank = bool(globals().get("ready_to_rank", False))

if "top_2_summary" in globals() and not top_2_summary.empty and "rank_status" in top_2_summary.columns:
    rank_status_value = str(top_2_summary.iloc[0]["rank_status"])
else:
    rank_status_value = "unknown"

meets_progress_threshold = current_progress_pct >= MIN_RETRIEVAL_PROGRESS_PCT
meets_rank_threshold = rank_status_value == "final"
meets_readiness_threshold = current_ready_to_rank

if not FORCE_EXPORT_DRAFT and not (
    meets_progress_threshold and meets_rank_threshold and meets_readiness_threshold
):
    raise RuntimeError(
        "Export blocked. Requirements not met: "
        f"retrieval_progress={current_progress_pct:.1f}% (min {MIN_RETRIEVAL_PROGRESS_PCT:.1f}%), "
        f"rank_status={rank_status_value}, "
        f"ready_to_rank={current_ready_to_rank}. "
        "Set FORCE_EXPORT_DRAFT=True only if you intentionally want a draft export."
    )

review_header = [
    "# AF Rotor Mechanisms - Review Draft",
    "",
    f"- Generated UTC: {RUN_METADATA.get('run_timestamp_utc', 'unknown') if 'RUN_METADATA' in globals() else 'unknown'}",
    f"- Shelf: {SHELF_NAME if 'SHELF_NAME' in globals() else 'unknown'}",
    f"- Rank status: {rank_status_value}",
    f"- Retrieval progress: {current_progress_pct:.1f}%",
    f"- Ready to rank: {current_ready_to_rank}",
    "",
    "> Note: If rank status is draft, additional cited retrieval evidence is still required.",
    "",
]

final_review_markdown = "\n".join(review_header) + draft_markdown
output_md_path = Path("af-rotor-mechanisms-review-draft.md")
output_md_path.write_text(final_review_markdown, encoding="utf-8")

print(f"Review draft written: {output_md_path.resolve()}")
output_md_path

RuntimeError: Export blocked. Requirements not met: retrieval_progress=70.0% (min 80.0%), rank_status=draft, ready_to_rank=False. Set FORCE_EXPORT_DRAFT=True only if you intentionally want a draft export.

### Export Final Draft As Markdown

Run this cell after the submission package cell to save a review-ready markdown document in the evaluation folder.

### How To Read This Output

- The markdown block is a structured starter draft, not a finished submission document.
- Evidence placeholder text indicates where citation-backed findings must be inserted before external use.
- The summary table is a completeness check to confirm all required package fields were populated.

## 24. Research Walkthrough Script (10-12 minutes)

1. **Minute 0-2:** Introduce problem: why rotor mechanisms remain debated in AF pathology.
2. **Minute 2-4:** Show retrieval prompts and explain evidence-gap separation.
3. **Minute 4-7:** Present five generated topics and associated endpoints.
4. **Minute 7-9:** Run weighted scoring and rank priorities.
5. **Minute 9-12:** Walk through top-2 blueprint and 12-month milestones.

Close with: this workflow is reproducible, auditable, and rapidly adaptable as the bookshelf grows.

In [56]:
# 25. Standalone GraphRAG query status monitor
from IPython.display import display
import pandas as pd

if "retrieval_capture_sheet" not in globals():
    if "discovery_query_plan" not in globals():
        raise RuntimeError(
            "retrieval_capture_sheet not found. Run the query-plan setup cells first."
        )
    retrieval_capture_sheet = discovery_query_plan[["mechanism", "prompt_name", "goal"]].copy()
    retrieval_capture_sheet["status"] = "pending"

monitor_view = retrieval_capture_sheet.copy()
monitor_view["status"] = (
    monitor_view.get("status", "pending")
    .fillna("pending")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"done": "complete", "completed": "complete", "in-progress": "partial"})
)

for required_status in ["pending", "partial", "complete"]:
    if required_status not in monitor_view["status"].unique():
        pass

status_counts = (
    monitor_view.groupby(["mechanism", "status"]).size().unstack(fill_value=0)
    .reindex(columns=["pending", "partial", "complete"], fill_value=0)
    .reset_index()
)
status_counts["total_queries"] = status_counts[["pending", "partial", "complete"]].sum(axis=1)
status_counts["completion_pct"] = (
    100 * (status_counts["complete"] + 0.5 * status_counts["partial"]) / status_counts["total_queries"].clip(lower=1)
).round(1)

overall_completion_pct = round(
    100 * (status_counts["complete"].sum() + 0.5 * status_counts["partial"].sum()) / status_counts["total_queries"].sum(),
    1,
    ) if status_counts["total_queries"].sum() else 0.0

def make_bar(value, width=24, filled="█", empty="░"):
    filled_width = int(round((max(min(value, 100.0), 0.0) / 100.0) * width))
    return filled * filled_width + empty * (width - filled_width)

status_counts["progress_visual"] = status_counts["completion_pct"].apply(
    lambda value: f"{make_bar(value)} {value:>5.1f}%"
 )
status_counts["queue_summary"] = status_counts.apply(
    lambda row: f"complete={int(row['complete'])}, partial={int(row['partial'])}, pending={int(row['pending'])}",
    axis=1,
)

ready_to_rank = bool(
    (status_counts["pending"].sum() == 0)
    and (status_counts["partial"].sum() == 0)
    and (len(status_counts) > 0)
)

print(f"Overall retrieval progress: {overall_completion_pct:.1f}%")
print("Safe-to-rank:", "YES" if ready_to_rank else "NO")

display(
    status_counts[[
        "mechanism",
        "progress_visual",
        "queue_summary",
        "completion_pct",
        "complete",
        "partial",
        "pending",
        "total_queries",
    ]].sort_values(["completion_pct", "mechanism"], ascending=[False, True])
)

if "requery_queue" in globals() and isinstance(requery_queue, pd.DataFrame) and not requery_queue.empty:
    print("\nMechanisms still flagged for requery:")
    display(requery_queue)

Overall retrieval progress: 0.0%
Safe-to-rank: NO


status,mechanism,progress_visual,queue_summary,completion_pct,complete,partial,pending,total_queries
0,Anisotropy gradients,░░░░░░░░░░░░░░░░░░░░░░░░ 0.0%,"complete=0, partial=0, pending=3",0.0,0,0,3,3
1,Autonomic heterogeneity,░░░░░░░░░░░░░░░░░░░░░░░░ 0.0%,"complete=0, partial=0, pending=3",0.0,0,0,3,3
2,Calcium instability,░░░░░░░░░░░░░░░░░░░░░░░░ 0.0%,"complete=0, partial=0, pending=3",0.0,0,0,3,3
3,Fibrosis topology,░░░░░░░░░░░░░░░░░░░░░░░░ 0.0%,"complete=0, partial=0, pending=3",0.0,0,0,3,3
4,Inflammation/coupling,░░░░░░░░░░░░░░░░░░░░░░░░ 0.0%,"complete=0, partial=0, pending=3",0.0,0,0,3,3



Mechanisms still flagged for requery:


,Mechanism,Candidate topic,Working hypothesis,Supporting evidence,Conflicting evidence,Supporting count,Conflicting count,Citation coverage,Average confidence,Evidence strength (1-5),Conflict penalty (0-2),Needs requery,Requery prompt,Key gap,Proposed endpoint,Retrieval timestamp UTC
0,Fibrosis topology,Fibrosis microarchitecture as rotor anchoring ...,Patchy intermediate-density fibrosis stabilize...,,,0,0,0.0,None,1.0,0.0,True,Re-query this mechanism with a citation-requir...,Need direct evidence to test whether patchy in...,Rotor core dwell time by fibrosis class,
1,Anisotropy gradients,Anisotropy gradients drive rotor drift toward ...,Local anisotropy gradients predict rotor meand...,,,0,0,0.0,None,1.0,0.0,True,Re-query this mechanism with a citation-requir...,Need direct evidence to test whether local ani...,Distance between predicted gradient zones and ...,
2,Calcium instability,Calcium alternans as a bridge from triggers to...,Increasing calcium instability increases waveb...,,,0,0,0.0,None,1.0,0.0,True,Re-query this mechanism with a citation-requir...,Need direct evidence to test whether increasin...,Wavebreak-to-rotor conversion rate,
3,Autonomic heterogeneity,Autonomic heterogeneity and dominant-frequency...,Autonomic gradients create local refractory di...,,,0,0,0.0,None,1.0,0.0,True,Re-query this mechanism with a citation-requir...,Need direct evidence to test whether autonomic...,Change in dominant frequency dispersion under ...,
4,Inflammation/coupling,Inflammation-linked connexin remodeling and tr...,Inflammatory surges alter coupling and transie...,,,0,0,0.0,None,1.0,0.0,True,Re-query this mechanism with a citation-requir...,Need direct evidence to test whether inflammat...,Association between biomarker surges and mappe...,
